In [0]:
# Fetch Snowflake secrets from retail-scope
snowflake_url = dbutils.secrets.get(
    scope="retail-scope",
    key="snowflake-url"
)

snowflake_user = dbutils.secrets.get(
    scope="retail-scope",
    key="snowflake-user"
)

snowflake_password = dbutils.secrets.get(
    scope="retail-scope",
    key="snowflake-password"
)

print("✅ Snowflake credentials loaded successfully from Azure Key Vault (retail-scope)!")


✅ Snowflake credentials loaded successfully from Azure Key Vault (retail-scope)!


In [0]:
# Snowflake connection options
sfOptions = {
    "sfURL": snowflake_url,
    "sfUser": snowflake_user,
    "sfPassword": snowflake_password,
    "sfDatabase": "RETAIL_CAPSTONE_DB",
    "sfSchema": "DBT",
    "sfWarehouse": "COMPUTE_WH",
    "sfRole": "ACCOUNTADMIN"
}

print("✅ Snowflake connection options configured!")


✅ Snowflake connection options configured!


In [0]:
# ============================================================
# Automatically builds ALL Gold Models:
# - dim_store
# - dim_date
# - fact_sales
# - fact_store_closures
# - agg_store_performance
# ============================================================

dbt_df = (
    spark.read
         .format("snowflake")
         .options(**sfOptions)
         .option(
             "query",
             """
             EXECUTE DBT PROJECT
             RETAIL_CAPSTONE_DB.DBT.DBT_PROJECT
             ARGS = 'run'
             """
         )
         .load()
)

print("✅ Success: Snowflake dbt pipeline executed successfully!")


✅ Success: Snowflake dbt pipeline executed successfully!


In [0]:
# ============================================================
# Shows recently built Gold tables with timestamps
# ============================================================

display(
    spark.read
         .format("snowflake")
         .options(**sfOptions)
         .option(
             "query",
             """
             SELECT
                 TABLE_NAME,
                 TABLE_TYPE,
                 LAST_ALTERED
             FROM RETAIL_CAPSTONE_DB.INFORMATION_SCHEMA.TABLES
             WHERE TABLE_SCHEMA = 'GOLD'
             ORDER BY LAST_ALTERED DESC
             """
         )
         .load()
)


TABLE_NAME,TABLE_TYPE,LAST_ALTERED
FACT_SALES,BASE TABLE,2026-09-21T12:40:25.211Z
DIM_DATE,BASE TABLE,2026-09-21T13:23:22.334Z
FACT_STORE_CLOSURES,BASE TABLE,2026-09-21T13:23:22.936Z
DIM_STORE,BASE TABLE,2026-09-21T13:23:22.342Z
AGG_STORE_PERFORMANCE,BASE TABLE,2026-09-18T22:45:06.469Z
